P4c — Clinical BERT Models: Colab execution companion

This execution-only copy runs M4 at learning rate 1e-4 for three epochs. M3 and the 2e-5 candidate remain in the local notebook.

In [1]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/HRP/ed-lbp-nlp-benchmark')
DATA_DIR = PROJECT_ROOT / 'results' / '03_experimental_setup'
OUTPUT_DIR = PROJECT_ROOT / 'results' / '04c_clinical_bert_models'
assert PROJECT_ROOT.is_dir(), PROJECT_ROOT
for name in ['modeling_dataset.csv', 'dataset_splits.csv']:
    assert (DATA_DIR / name).is_file(), DATA_DIR / name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)
print('Drive free space (GB):', round(shutil.disk_usage(OUTPUT_DIR).free / 1e9, 2))

Mounted at /content/drive
Data directory: /content/drive/MyDrive/HRP/ed-lbp-nlp-benchmark/results/03_experimental_setup
Output directory: /content/drive/MyDrive/HRP/ed-lbp-nlp-benchmark/results/04c_clinical_bert_models
Drive free space (GB): 15.8


## 2. Development Dataset Preparation

I use the same modeling dataset and fixed patient-grouped partitions as in the previous notebooks to ensure a consistent comparison across models.

Both M3 and M4 use the cleaned chief complaint text as input and the ICD-derived binary LBP label as the prediction target. The training set is used for model development, and the validation set is used for model selection and classification threshold selection. The test set remains held out for the final comparative evaluation.

In [2]:
from pathlib import Path
import pandas as pd

# Set the data directory for the current environment
assert DATA_DIR.is_dir()

# Load the modeling dataset and fixed partition assignments
df = pd.read_csv(
    DATA_DIR / "modeling_dataset.csv",
    keep_default_na=False
)

splits = pd.read_csv(DATA_DIR / "dataset_splits.csv")

# Retain development partitions only
dev_splits = splits[splits["split"].isin(["train", "validation"])]

dev_df = df.merge(
    dev_splits,
    on="stay_id",
    how="inner",
    validate="one_to_one"
)

train_df = dev_df[dev_df["split"] == "train"].copy().reset_index(drop=True)
val_df = dev_df[dev_df["split"] == "validation"].copy().reset_index(drop=True)

# Prepare model inputs and reference labels
X_train = train_df["chiefcomplaint_clean"]
y_train = train_df["lbp_label"]

X_val = val_df["chiefcomplaint_clean"]
y_val = val_df["lbp_label"]

# Check input integrity
assert len(dev_df) == len(dev_splits)
assert dev_df["chiefcomplaint_clean"].notna().all()
assert dev_df["chiefcomplaint_clean"].str.strip().ne("").all()
assert dev_df["lbp_label"].isin([0, 1]).all()

# Summarize development partitions
summary = dev_df.groupby("split").agg(
    encounters=("stay_id", "size"),
    patients=("subject_id", "nunique"),
    positives=("lbp_label", "sum")
)

summary["negatives"] = summary["encounters"] - summary["positives"]
summary["prevalence (%)"] = (
    summary["positives"] / summary["encounters"] * 100
).round(2)

display(summary)

,encounters,patients,positives,negatives,prevalence (%)
split,,,,,
train,295877,143119,11429,284448,3.86
validation,42268,20483,1632,40636,3.86


## 3. Clinical BERT Configuration (M4 Only)

I use the pretrained Bio_ClinicalBERT checkpoint and its original tokenizer for both M3 and M4. Given the short length of ED chief complaints, I set the maximum sequence length to 64 tokens and use dynamic padding during batch processing to avoid unnecessary computation.

To support execution across different computing environments, I automatically select CUDA, MPS, or CPU based on device availability. Before processing the full dataset, I run a small pilot to verify tokenization, model output dimensions, and computational feasibility.

In [3]:
import random
import numpy as np
import torch
import transformers
from time import perf_counter
from transformers import AutoTokenizer

# Set random seeds
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Select the available computing device
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Match the original local FP32 run as closely as practical.
assert device.type == "cuda", "Select a GPU runtime before continuing."
use_fp16 = False

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Device: {device}")
print(f"FP16 mixed precision: {use_fp16}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.14.1+cu130
Transformers version: 5.22.1
Device: cuda
FP16 mixed precision: False
GPU: Tesla T4


In [4]:
from contextlib import nullcontext
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 64
TARGET_LR = 1e-4
THRESHOLD_GRID = [i / 100 for i in range(5, 100, 5)]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

print("Checkpoint:", MODEL_NAME)
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Checkpoint: emilyalsentzer/Bio_ClinicalBERT
Vocabulary size: 28996


4. M3 — skipped in this execution copy
Frozen embeddings and logistic regression are handled in the local notebook.

5. M4 — Fine-tuned Clinical BERT
5.1 Model Configuration
While M3 uses Clinical BERT as a frozen feature extractor, M4 updates the pretrained encoder and classification head through supervised learning. This allows the model to adapt its contextual representations to the ICD-defined LBP classification task.

I use the same pretrained checkpoint and tokenizer as M3, with a binary classification head. To account for class imbalance, I use weighted cross-entropy loss. The model is trained using AdamW with two learning rate candidates and three epochs per candidate. Model and classification threshold selection are based on positive-class F1 on the validation set.

In [5]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

# Fine-tuning configuration
NUM_LABELS = 2
LR_GRID = [2e-5, 1e-4]
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 32

# Initialize an independent model from the pretrained checkpoint
def initialize_m4_model():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    )

    return model.to(device)

# Display fine-tuning configuration
m4_config = {
    "checkpoint": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "max_length": MAX_LENGTH,
    "learning_rates": LR_GRID,
    "epochs": NUM_EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW",
    "device": str(device),
    "fp16": use_fp16,
    "seed": SEED
}

display(pd.Series(m4_config, name="M4 Configuration"))

,M4 Configuration
checkpoint,emilyalsentzer/Bio_ClinicalBERT
num_labels,2
max_length,64
learning_rates,"[2e-05, 0.0001]"
epochs,3
batch_size,32
weight_decay,0.01
optimizer,AdamW
device,cuda
fp16,False


5.2 Dataset and DataLoader
I prepare the existing training and validation partitions for supervised BERT fine-tuning using PyTorch datasets and data loaders. Chief complaints are tokenized in batches with dynamic padding and a maximum sequence length of 64 tokens.

Training batches are shuffled, while validation batches preserve the original encounter order. To address class imbalance, I calculate balanced class weights using only the training labels. These weights will be used in the cross-entropy loss function during fine-tuning.

In [6]:
from torch.utils.data import Dataset, DataLoader

class ChiefComplaintDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

        assert len(self.texts) == len(self.labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": int(self.labels[idx])
        }


# Tokenize and dynamically pad each batch
def collate_batch(batch):
    texts = [item["text"] for item in batch]
    labels = [item["label"] for item in batch]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    inputs["labels"] = torch.tensor(
        labels,
        dtype=torch.long
    )

    return inputs


# Create development datasets
train_dataset = ChiefComplaintDataset(X_train, y_train)
val_dataset = ChiefComplaintDataset(X_val, y_val)

print(f"Training dataset: {len(train_dataset):,}")
print(f"Validation dataset: {len(val_dataset):,}")

Training dataset: 295,877
Validation dataset: 42,268


In [7]:
def create_dataloaders(batch_size=TRAIN_BATCH_SIZE, seed=SEED):

    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_batch,
        generator=generator,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=(device.type == "cuda")
    )

    return train_loader, val_loader


train_loader, val_loader = create_dataloaders()

print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

# Inspect batch dimensions without displaying clinical text
sample_batch = next(iter(val_loader))

for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

Training batches: 9,247
Validation batches: 1,321
input_ids: torch.Size([32, 15])
token_type_ids: torch.Size([32, 15])
attention_mask: torch.Size([32, 15])
labels: torch.Size([32])


In [8]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate balanced weights using training labels only
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train.to_numpy()
)

# Convert weights to a PyTorch tensor
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

# Summarize class weights
weight_summary = pd.DataFrame({
    "label": [0, 1],
    "training_count": [
        (y_train == 0).sum(),
        (y_train == 1).sum()
    ],
    "class_weight": class_weights
})

display(weight_summary.round(4))

,label,training_count,class_weight
0,0,284448,0.5201
1,1,11429,12.9441


5.3 Training and Validation Functions
I define reusable training and validation functions for supervised Clinical BERT fine-tuning. During training, weighted cross-entropy accounts for class imbalance, while AdamW updates both the encoder and classification head. Gradient clipping is applied to improve numerical stability.

After each epoch, the model is evaluated on the fixed validation set. I apply the same classification threshold grid used for the previous supervised models and select the configuration that achieves the highest positive-class validation F1.

In [9]:
from torch import nn

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    scaler
):
    model.train()

    total_loss = 0.0
    total_weight = 0.0

    for batch in tqdm(dataloader, desc="Training"):

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        labels = batch.pop("labels")

        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        with torch.autocast(
            "cuda", dtype=torch.float16
        ) if use_fp16 else nullcontext():

            outputs = model(**batch)
            loss = criterion(outputs.logits, labels)

        # Backward pass
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # Update model parameters
        scaler.step(optimizer)
        scaler.update()

        # Aggregate weighted training loss
        batch_weight = criterion.weight[labels].sum().item()

        total_loss += loss.item() * batch_weight
        total_weight += batch_weight

    return total_loss / total_weight

In [10]:
def predict_m4_probabilities(model, dataloader):

    model.eval()
    all_probs = []

    with torch.inference_mode():

        for batch in tqdm(
            dataloader,
            desc="Validation"
        ):

            batch = {
                k: v.to(device)
                for k, v in batch.items()
                if k != "labels"
            }

            with torch.autocast(
                "cuda", dtype=torch.float16
            ) if use_fp16 else nullcontext():

                outputs = model(**batch)

            # Obtain positive-class probabilities
            probs = torch.softmax(
                outputs.logits.float(),
                dim=1
            )[:, 1]

            all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs)

In [11]:
def evaluate_thresholds(y_true, probs):

    assert len(y_true) == len(probs)
    assert np.isfinite(probs).all()

    records = []

    for threshold in THRESHOLD_GRID:

        predictions = (probs >= threshold).astype(int)

        records.append({
            "threshold": threshold,
            "f1": f1_score(
                y_true,
                predictions,
                zero_division=0
            )
        })

    results = pd.DataFrame(records)

    results = results.sort_values(
        by=["f1", "threshold"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return results

5.4 Training Pilot
Before full fine-tuning, I run a small training pilot to verify that the forward pass, backward pass, weighted loss, and optimizer updates work correctly on the selected device.

The pilot uses 100 batches from the training set to estimate computational requirements and assess whether the planned batch size is feasible. The pilot model is discarded afterward, and each formal training run starts independently from the original pretrained checkpoint.

In [12]:
from itertools import islice

PILOT_BATCHES = 100

# Initialize a temporary model
pilot_model = initialize_m4_model()

# Configure weighted loss and optimizer
pilot_criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

pilot_optimizer = AdamW(
    pilot_model.parameters(),
    lr=TARGET_LR,
    weight_decay=WEIGHT_DECAY
)

pilot_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_fp16
)

# Create an independent training data loader
pilot_train_loader, _ = create_dataloaders()

print(f"Device: {device}")
print(f"Batch size: {TRAIN_BATCH_SIZE}")
print(f"Pilot batches: {PILOT_BATCHES}")
print(f"Total parameters: {sum(p.numel() for p in pilot_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in pilot_model.parameters() if p.requires_grad):,}")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

Device: cuda
Batch size: 32
Pilot batches: 100
Total parameters: 108,311,810
Trainable parameters: 108,311,810


In [13]:
# Limit the pilot to the first 100 shuffled training batches
pilot_batches = islice(
    pilot_train_loader,
    PILOT_BATCHES
)

synchronize_device()
start = perf_counter()

pilot_loss = train_one_epoch(
    model=pilot_model,
    dataloader=pilot_batches,
    optimizer=pilot_optimizer,
    criterion=pilot_criterion,
    scaler=pilot_scaler
)

synchronize_device()
elapsed = perf_counter() - start

pilot_samples = PILOT_BATCHES * TRAIN_BATCH_SIZE
throughput = pilot_samples / elapsed

assert np.isfinite(pilot_loss)

print(f"\nPilot batches: {PILOT_BATCHES}")
print(f"Pilot encounters: {pilot_samples:,}")
print(f"Training loss: {pilot_loss:.4f}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Throughput: {throughput:.2f} encounters/second")
print("Training pilot: PASSED")

Training: 0it [00:00, ?it/s]


Pilot batches: 100
Pilot encounters: 3,200
Training loss: 0.6873
Elapsed time: 15.79 seconds
Throughput: 202.60 encounters/second
Training pilot: PASSED


In [14]:
# Estimate runtime based on observed pilot throughput
epoch_seconds = len(train_dataset) / throughput

run_seconds = epoch_seconds * NUM_EPOCHS

print(f"Estimated one epoch: {epoch_seconds / 60:.2f} min")
print(f"Estimated target run (3 epochs): {run_seconds / 60:.2f} min")

Estimated one epoch: 24.34 min
Estimated target run (3 epochs): 73.02 min


In [15]:
import gc

# Discard the pilot model and optimizer
del pilot_model
del pilot_optimizer
del pilot_criterion
del pilot_scaler
del pilot_batches
del pilot_train_loader

gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

print("Pilot resources released.")

Pilot resources released.


5.5 Model Training and Selection
Each learning rate candidate is trained independently from the original pretrained checkpoint for three epochs. After each epoch, I evaluate the model on the fixed validation set and search the predefined classification threshold grid.

To preserve intermediate results, I save the training history, threshold search results, and the best-performing model in the mounted Drive output directory. I also retain the most recent training checkpoint so that an interrupted run can resume from the last completed epoch.

The final model is selected based on positive-class validation F1, with ties resolved by preferring a smaller learning rate, an earlier epoch, and a higher classification threshold. No additional refitting is performed after model selection.

In [16]:
import json
import gc

M4_OUTPUT_DIR = OUTPUT_DIR
M4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Save checkpoints without overwriting a valid file
# until the new file has been written successfully
def save_checkpoint(state, path):
    temp_path = path.with_suffix(".tmp")
    torch.save(state, temp_path)
    temp_path.replace(path)


# Capture random states for training resumption
def capture_rng_state(generator):
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "dataloader": generator.get_state()
    }

    if device.type == "cuda":
        state["cuda"] = torch.cuda.get_rng_state_all()
    elif device.type == "mps":
        state["mps"] = torch.mps.get_rng_state()

    return state


# Restore random states after checkpoint loading
def restore_rng_state(state, generator):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    generator.set_state(state["dataloader"])

    if device.type == "cuda":
        torch.cuda.set_rng_state_all(state["cuda"])
    elif device.type == "mps":
        torch.mps.set_rng_state(state["mps"])

In [17]:
def save_run_results(run_dir, history, search_records, best_config):

    # Save epoch-level training history
    pd.DataFrame(history).to_csv(
        run_dir / "training_history.csv",
        index=False
    )

    # Save all validation threshold candidates
    pd.DataFrame(search_records).to_csv(
        run_dir / "threshold_search_results.csv",
        index=False
    )

    # Save the best configuration for this run
    with open(run_dir / "best_config.json", "w") as f:
        json.dump(best_config, f, indent=4)

In [18]:
def train_m4_candidate(learning_rate, resume=False):

    assert learning_rate in LR_GRID

    # Create a separate directory for each learning rate
    run_dir = M4_OUTPUT_DIR / f"m4_lr_{learning_rate:.0e}"
    run_dir.mkdir(parents=True, exist_ok=True)

    last_path = run_dir / "last_checkpoint.pt"
    best_path = run_dir / "best_model.pt"

    if last_path.exists() and not resume:
        raise FileExistsError(
            "Existing checkpoint found. Use resume=True "
            "to continue this training run."
        )

    if resume and not last_path.exists():
        raise FileNotFoundError("No checkpoint available to resume.")

    # Initialize model and training components
    model = initialize_m4_model()

    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_fp16
    )

    train_loader, val_loader = create_dataloaders()

    start_epoch = 1
    history = []
    search_records = []
    best_config = None

    # Restore the last completed epoch if requested
    if resume:

        checkpoint = torch.load(
            last_path,
            map_location="cpu",
            weights_only=False
        )

        assert checkpoint["learning_rate"] == learning_rate

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

        history = checkpoint["history"]
        search_records = checkpoint["search_records"]
        best_config = checkpoint["best_config"]

        start_epoch = checkpoint["epoch"] + 1

        restore_rng_state(
            checkpoint["rng_state"],
            train_loader.generator
        )

        # Recover the best model if saving was interrupted
        if best_config["epoch"] == checkpoint["epoch"]:
            save_checkpoint(
                model.state_dict(),
                best_path
            )

        save_run_results(
            run_dir, history, search_records, best_config
        )

        del checkpoint

        print(f"Resuming from epoch {start_epoch}")

    # Train and validate after each epoch
    for epoch in range(start_epoch, NUM_EPOCHS + 1):

        print(
            f"\nLearning rate: {learning_rate:.0e} | "
            f"Epoch {epoch}/{NUM_EPOCHS}"
        )

        synchronize_device()
        start = perf_counter()

        # Training
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler
        )

        assert np.isfinite(train_loss)

        synchronize_device()
        train_seconds = perf_counter() - start

        # Validation
        start = perf_counter()

        val_probs = predict_m4_probabilities(
            model,
            val_loader
        )

        threshold_results = evaluate_thresholds(
            y_val,
            val_probs
        )

        val_seconds = perf_counter() - start

        # Retain all threshold search results
        threshold_results["learning_rate"] = learning_rate
        threshold_results["epoch"] = epoch

        search_records.extend(
            threshold_results.to_dict("records")
        )

        # Identify the best threshold for this epoch
        epoch_best = threshold_results.iloc[0]

        epoch_config = {
            "learning_rate": learning_rate,
            "epoch": epoch,
            "threshold": float(epoch_best["threshold"]),
            "f1": float(epoch_best["f1"])
        }

        history.append({
            **epoch_config,
            "training_loss": float(train_loss),
            "training_seconds": train_seconds,
            "validation_seconds": val_seconds
        })

        # Prefer the earlier epoch when F1 is tied
        improved = (
            best_config is None
            or epoch_config["f1"] > best_config["f1"]
        )

        if improved:
            best_config = epoch_config.copy()

        # Save the latest completed training state
        save_checkpoint(
            {
                "learning_rate": learning_rate,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "rng_state": capture_rng_state(
                    train_loader.generator
                ),
                "history": history,
                "search_records": search_records,
                "best_config": best_config
            },
            last_path
        )

        # Save the best-performing model weights
        if improved:
            save_checkpoint(
                model.state_dict(),
                best_path
            )

            print("New best checkpoint saved.")

        # Persist intermediate results
        save_run_results(
            run_dir,
            history,
            search_records,
            best_config
        )

        print(f"Training loss: {train_loss:.4f}")
        print(f"Validation F1: {epoch_config['f1']:.4f}")
        print(f"Best threshold: {epoch_config['threshold']:.2f}")
        print(f"Training time: {train_seconds / 60:.2f} min")
        print(f"Validation time: {val_seconds / 60:.2f} min")
        print(f"Epoch {epoch} results saved.")

    # Release training resources
    del model, optimizer, criterion, scaler
    del train_loader, val_loader

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

    return best_config

In [19]:
from tempfile import TemporaryDirectory

# Initialize temporary training components
smoke_model = initialize_m4_model()

smoke_optimizer = AdamW(
    smoke_model.parameters(),
    lr=TARGET_LR,
    weight_decay=WEIGHT_DECAY
)

smoke_criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

smoke_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_fp16
)

smoke_loader, _ = create_dataloaders()

# Run two batches to initialize optimizer states
smoke_loss = train_one_epoch(
    smoke_model,
    islice(smoke_loader, 2),
    smoke_optimizer,
    smoke_criterion,
    smoke_scaler
)

assert np.isfinite(smoke_loss)

# Test checkpoint saving and loading
with TemporaryDirectory(dir=M4_OUTPUT_DIR) as temp_dir:

    smoke_path = Path(temp_dir) / "smoke_checkpoint.pt"

    save_checkpoint(
        {
            "epoch": 1,
            "model_state_dict": smoke_model.state_dict(),
            "optimizer_state_dict": smoke_optimizer.state_dict(),
            "scaler_state_dict": smoke_scaler.state_dict(),
            "rng_state": capture_rng_state(
                smoke_loader.generator
            )
        },
        smoke_path
    )

    print(f"Checkpoint size: {smoke_path.stat().st_size / 1e6:.2f} MB")

    checkpoint = torch.load(
        smoke_path,
        map_location="cpu",
        weights_only=False
    )

    # Verify saved model weights
    assert all(
        torch.equal(
            param.detach().cpu(),
            checkpoint["model_state_dict"][name]
        )
        for name, param in smoke_model.state_dict().items()
    )

    # Restore training states
    smoke_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    smoke_optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    smoke_scaler.load_state_dict(
        checkpoint["scaler_state_dict"]
    )

    restore_rng_state(
        checkpoint["rng_state"],
        smoke_loader.generator
    )

    assert checkpoint["epoch"] == 1

    print("Model state: PASSED")
    print("Optimizer state loading: PASSED")
    print("Scaler state loading: PASSED")
    print("RNG state restoration: PASSED")

# Release temporary resources
del checkpoint
del smoke_model, smoke_optimizer
del smoke_criterion, smoke_scaler, smoke_loader

gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()
elif device.type == "mps":
    torch.mps.empty_cache()

print("Checkpoint smoke test: PASSED")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

Training: 0it [00:00, ?it/s]

Checkpoint size: 1300.02 MB
Model state: PASSED
Optimizer state loading: PASSED
Scaler state loading: PASSED
RNG state restoration: PASSED
Checkpoint smoke test: PASSED


5.5.2 Independent Training Run — 1e-4 only
Start from the pretrained checkpoint or resume the existing Colab run from its last completed epoch. Do not resume a local MPS checkpoint here.

In [20]:
# Save environment metadata alongside this candidate only.
import platform
import sklearn
run_dir = M4_OUTPUT_DIR / f"m4_lr_{TARGET_LR:.0e}"
run_dir.mkdir(parents=True, exist_ok=True)
environment = {**m4_config, "target_learning_rate": TARGET_LR,
               "python": platform.python_version(), "torch": str(torch.__version__),
               "transformers": transformers.__version__, "sklearn": sklearn.__version__,
               "gpu": torch.cuda.get_device_name(0)}
metadata_path = run_dir / "colab_environment.json"
if metadata_path.exists():
    previous = json.loads(metadata_path.read_text())
    assert previous == environment, "Environment changed; review before resuming."
else:
    assert not (run_dir / "last_checkpoint.pt").exists(), "Existing checkpoint without Colab provenance; review before resuming."
    metadata_path.write_text(json.dumps(environment, indent=2))

m4_result_lr2 = train_m4_candidate(
    learning_rate=TARGET_LR,
    resume=(run_dir / "last_checkpoint.pt").exists()
)
print(m4_result_lr2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec


Learning rate: 1e-04 | Epoch 1/3


Training:   0%|          | 0/9247 [00:00<?, ?it/s]

Validation:   0%|          | 0/1321 [00:00<?, ?it/s]

New best checkpoint saved.
Training loss: 0.7152
Validation F1: 0.0744
Best threshold: 0.50
Training time: 18.44 min
Validation time: 0.58 min
Epoch 1 results saved.

Learning rate: 1e-04 | Epoch 2/3


... (2 outputs left out)

Training loss: 0.7091
Validation F1: 0.0744
Best threshold: 0.40
Training time: 18.27 min
Validation time: 0.58 min
Epoch 2 results saved.

Learning rate: 1e-04 | Epoch 3/3


Training:   0%|          | 0/9247 [00:00<?, ?it/s]

Validation:   0%|          | 0/1321 [00:00<?, ?it/s]

Training loss: 0.7081
Validation F1: 0.0744
Best threshold: 0.50
Training time: 18.37 min
Validation time: 0.58 min
Epoch 3 results saved.
{'learning_rate': 0.0001, 'epoch': 1, 'threshold': 0.5, 'f1': 0.0743507972665148}


5.6 Candidate results
Review validation results only. Compare both candidates in the local notebook after both runs finish.

In [ ]:
history = pd.read_csv(run_dir / "training_history.csv")
display(history)
assert history["epoch"].tolist() == list(range(1, NUM_EPOCHS + 1))
print("Candidate artifacts:")
for path in sorted(run_dir.iterdir()):
    print(path.name, f"{path.stat().st_size / 1e6:.2f} MB")